<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/MappingReq_US_Gaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================================
#  FINAL FIXED: English vs Gherkin + Manual Override + Gap Report
#  NO NLTK ERRORS — punkt_tab downloaded
#  Google Colab — Just upload 2 files
# ========================================================

# --- INSTALL PACKAGES ---
import subprocess, sys
def install(p): subprocess.check_call([sys.executable, "-m", "pip", "install", p], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Installing packages...")
install("pandas"); install("openpyxl"); install("fuzzywuzzy"); install("python-levenshtein"); install("nltk")

import pandas as pd, io, base64, ipywidgets as widgets, re
from IPython.display import display, HTML, clear_output
from fuzzywuzzy import fuzz
import nltk

# --- DOWNLOAD NLTK DATA (FIXED) ---
print("Downloading NLTK data (punkt_tab, stopwords)...")
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)  # fallback

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# --- TEXT CLEANING ---
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(text).lower())
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()  # fallback
    return [t for t in tokens if t not in stop_words and len(t) > 2]

# --- ENGLISH vs GHERKIN COMPARISON ---
def compare_english_vs_gherkin(req_text, gherkin):
    req_clean = clean_text(req_text)
    gherkin_clean = clean_text(gherkin)

    # Jaccard
    overlap = len(set(req_clean) & set(gherkin_clean))
    jaccard = overlap / len(set(req_clean) | set(gherkin_clean)) if req_clean or gherkin_clean else 0

    # Fuzzy
    fuzzy_score = fuzz.partial_ratio(req_text.lower(), gherkin.lower()) / 100

    # Keywords
    keywords = ['must', 'shall', 'button', 'lookup', 'mandatory', 'system generated', 'auto-populated', 'save', 'edit', 'display']
    kw_boost = sum(1 for kw in keywords if kw in req_text.lower() and kw in gherkin.lower()) * 0.05

    final_score = int((jaccard * 60) + (fuzzy_score * 30) + (kw_boost * 100))
    final_score = min(final_score, 100)

    color = "green" if final_score >= 90 else "orange" if final_score >= 70 else "red"
    return final_score, f"<span style='color:{color}'>Score: {final_score}</span>", overlap

# --- COLUMN DETECTION ---
def find_column(df, candidates):
    for c in candidates:
        if any(col.lower() == c.lower() for col in df.columns):
            return next(col for col in df.columns if col.lower() == c.lower())
    return None

# --- MAIN MATCHING + COMPARISON ---
def match_and_compare(row, stories_df, manual_col):
    req_text = row['Req_Text']
    manual = str(row.get(manual_col, '')).strip()

    # 1. MANUAL OVERRIDE
    if manual and manual not in ['nan', '', 'None']:
        ids = [x.strip() for x in re.split(r'[/,;\s]+', manual) if x.strip() and len(x.strip()) >= 4]
        valid = [id for id in ids if id in stories_df['detected_id'].astype(str).values]
        if valid:
            results = []
            for vid in valid:
                story = stories_df[stories_df['detected_id'] == vid].iloc[0]
                score, _, _ = compare_english_vs_gherkin(req_text, story['detected_ac'])
                gherkin_snippet = " ".join(story['detected_ac'].split()[:30]) + ("..." if len(story['detected_ac'].split()) > 30 else "")
                results.append(f"{vid} (Score: {score})")
            return ", ".join(results), "MANUAL", gherkin_snippet

    # 2. AUTO MATCH
    req_lower = req_text.lower()
    hierarchy = str(row['Hierarchy']).lower()
    tags = str(row['Tags']).lower()
    req_ids = re.findall(r'\b([a-z]{2}\.[a-z]{2}\.[a-z]{3}\.\d{4})\b', req_lower) + re.findall(r'\b(\d{5,})\b', req_lower + tags)
    req_ids = list(set(req_ids))

    best_match = None
    best_score = 0
    best_gherkin = ""

    for _, story in stories_df.iterrows():
        story_id = story['detected_id']
        gherkin = str(story['detected_ac']).lower()

        # ID match
        if any(rid in gherkin for rid in req_ids):
            score, _, _ = compare_english_vs_gherkin(req_text, story['detected_ac'])
            gherkin_snippet = " ".join(story['detected_ac'].split()[:30]) + ("..." if len(story['detected_ac'].split()) > 30 else "")
            return f"{story_id} (Score: {score})", "AUTO-ID", gherkin_snippet

        # Hierarchy + Tag + Overlap
        hier_match = any(part in gherkin for part in hierarchy.split("→") if len(part) > 3)
        tag_list = [t.strip() for t in tags.split(",") if len(t.strip()) > 2]
        tag_match = any(t in gherkin for t in tag_list)
        overlap_words = len(set(clean_text(req_text)) & set(clean_text(gherkin)))

        score = (hier_match * 30) + (tag_match * 40) + (overlap_words * 5)
        if score > best_score:
            best_score = score
            best_match = story_id
            best_gherkin = " ".join(story['detected_ac'].split()[:30]) + ("..." if len(story['detected_ac'].split()) > 30 else "")

    if best_score >= 50:
        score, _, _ = compare_english_vs_gherkin(req_text, stories_df[stories_df['detected_id'] == best_match]['detected_ac'].iloc[0])
        return f"{best_match} (Score: {score})", "AUTO", best_gherkin

    return "GAP", "GAP", ""

# --- UI ---
story_upload = widgets.FileUpload(accept='.xlsx,.csv,.xls', description="User Stories")
req_upload = widgets.FileUpload(accept='.xlsx,.csv,.xls', description="DevOps Requirements")
status = widgets.Output(); output = widgets.Output()
uploaded = {"stories": False, "reqs": False}

def on_story(change):
    if story_upload.value:
        uploaded["stories"] = True
        with status: clear_output(); print("User Stories uploaded")
        check_run()
def on_req(change):
    if req_upload.value:
        uploaded["reqs"] = True
        with status: clear_output(); print("DevOps Requirements uploaded")
        check_run()

def check_run():
    if uploaded["stories"] and uploaded["reqs"]:
        with output: clear_output(); print("RUNNING FINAL ANALYSIS + COMPARISON...\n"); run_analysis()

def run_analysis():
    try:
        # --- READ STORIES ---
        s_info = list(story_upload.value.values())[0]
        s_io = io.BytesIO(s_info['content'])
        stories_df = pd.read_excel(s_io) if s_info['metadata']['name'].lower().endswith(('.xlsx','.xls')) else pd.read_csv(s_io)
        id_col = find_column(stories_df, ["ID", "Work Item ID"])
        ac_col = find_column(stories_df, ["Acceptance Criteria"])
        if not id_col or not ac_col:
            print("ERROR: Missing ID or Acceptance Criteria in User Stories")
            return
        stories_df['detected_id'] = stories_df[id_col].astype(str)
        stories_df['detected_ac'] = stories_df[ac_col].fillna("")

        # --- READ REQUIREMENTS ---
        r_info = list(req_upload.value.values())[0]
        r_io = io.BytesIO(r_info['content'])
        reqs_df = pd.read_excel(r_io) if r_info['metadata']['name'].lower().endswith(('.xlsx','.xls')) else pd.read_csv(r_io)
        text_col = find_column(reqs_df, ["Requirement Text", "Req_Text"])
        tag_col = find_column(reqs_df, ["Tags"])
        title_cols = [c for c in reqs_df.columns if re.match(r"Title\s*\d", c, re.I)]
        manual_col = find_column(reqs_df, ["User Story", "To confirm"])

        reqs_df['Req_Text'] = reqs_df[text_col].fillna("")
        reqs_df['Tags'] = reqs_df[tag_col].fillna("") if tag_col else ""
        reqs_df['Hierarchy'] = reqs_df.apply(lambda row: " → ".join([str(row[c]).strip() for c in title_cols if str(row[c]).strip() and str(row[c]).lower() not in ["nan", ""]]), axis=1)

        # --- MATCH & COMPARE ---
        results = []
        for _, row in reqs_df.iterrows():
            covered, method, gherkin = match_and_compare(row, stories_df, manual_col)
            results.append({
                'Req_Text': row['Req_Text'],
                'Hierarchy': row['Hierarchy'],
                'Manual_User_Story': row.get(manual_col, ''),
                'Covered_By': covered,
                'Match_Method': method,
                'Gherkin_Snippet': gherkin,
                'Status': "COVERED" if covered != "GAP" else "GAP"
            })
        result_df = pd.DataFrame(results)

        # --- REPORT ---
        gaps = result_df[result_df['Status'] == 'GAP']
        print(f"FINAL REPORT")
        print(f"Total: {len(result_df)} | Covered: {len(result_df)-len(gaps)} | GAPS: {len(gaps)}")

        if len(gaps) > 0:
            print(f"\nREAL GAPS (NO COVERAGE):")
            display(gaps[['Req_Text', 'Hierarchy', 'Manual_User_Story']].head(20))
        else:
            print("\nNO GAPS! FULL COVERAGE!")

        # --- DOWNLOAD ---
        buf = io.BytesIO()
        result_df.to_excel(buf, index=False, engine='openpyxl')
        buf.seek(0)
        link = f'<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{base64.b64encode(buf.read()).decode()}" download="FINAL_ENGLISH_VS_GHERKIN_REPORT.xlsx">DOWNLOAD FULL REPORT</a>'
        display(HTML(f"<br><strong>{link}</strong>"))

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback; traceback.print_exc()

# --- OBSERVE ---
story_upload.observe(on_story, 'value')
req_upload.observe(on_req, 'value')

# --- UI ---
display(HTML("<h3>Upload → Auto-Run Final Truth + Comparison</h3>"))
display(HTML("<b>1. User Stories</b>"), story_upload)
display(HTML("<b>2. DevOps Requirements</b>"), req_upload)
display(HTML("<b>Status:</b>"), status)
display(HTML("<b>Output:</b>"), output)